
# Mission C - Collecte des prix du Gazole avec Python

## Projet : OptiFleet Logistics

### Objectif

Interroger l'API publique des prix des carburants pour récupérer toutes les stations-service du département 69,
y compris celles dont le prix du Gazole est manquant.

Les contrôles qualité et les filtres métier seront appliqués ensuite avec Pandas.

### Étapes du notebook

1. Importer la bibliothèque requests.
2. Envoyer une requête HTTP GET.
3. Vérifier la réponse de l'API.
4. Lire les données JSON.
5. Transformer les résultats en DataFrame Pandas.
6. Récupérer toutes les stations avec la pagination.
7. Effectuer les premiers contrôles de qualité.

### Source

API Explore v2.1 - Prix des carburants en France,
Flux instantané v2.

### Périmètre initial

Département du Rhône (code 69).

Le périmètre final du projet sera la Métropole de Lyon.


In [2]:
# Vérifier la bibliothèque requests
import json
from datetime import datetime, timezone
from pathlib import Path
import requests

print("Bibliothèque requests disponible !")
print("Version :", requests.__version__)

Bibliothèque requests disponible !
Version : 2.32.5


In [3]:
URL = (
    "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/"
    "prix-des-carburants-en-france-flux-instantane-v2/records"
)

params = {
    "where": 'code_departement="69"',
    "limit": 5
}

reponse = requests.get(URL, params=params, timeout=30)

print("Code HTTP :", reponse.status_code)
print("URL appelée :", reponse.url)

Code HTTP : 200
URL appelée : https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/prix-des-carburants-en-france-flux-instantane-v2/records?where=code_departement%3D%2269%22&limit=5


In [4]:
reponse.raise_for_status()

donnees = reponse.json()

print("Type :", type(donnees))
print("Clés :", donnees.keys())
print("Total :", donnees["total_count"])
print("Stations reçues :", len(donnees["results"]))

Type : <class 'dict'>
Clés : dict_keys(['total_count', 'results'])
Total : 200
Stations reçues : 5


## C2 - Pagination automatique des stations-service

Objectif : récupérer toutes les stations du département 69,
y compris celles sans prix Gazole renseigné.

L'API autorise au maximum 100 résultats par page.

Une boucle while permettra de récupérer automatiquement
toutes les pages, quel que soit le nombre de stations.

In [5]:
# Paramètres de départ
taille_page = 100
offset = 0

toutes_stations = []
total_annonce = None

while True:

    # Construire les paramètres de la page courante
    params_page = {
        "where": 'code_departement="69"',
        "order_by": "id ASC",
        "limit": taille_page,
        "offset": offset
    }

    # Envoyer la requête
    reponse_page = requests.get(
        URL,
        params=params_page,
        timeout=30
    )

    # Vérifier la réponse HTTP
    reponse_page.raise_for_status()

    # Convertir le JSON
    donnees_page = reponse_page.json()

    # Lire le total annoncé lors de la première requête
    if total_annonce is None:
        total_annonce = donnees_page["total_count"]

    # Récupérer les stations de cette page
    stations_page = donnees_page["results"]

    # Ajouter les stations à notre liste générale
    toutes_stations.extend(stations_page)

    print(
        f"Offset {offset} : "
        f"{len(stations_page)} stations reçues"
    )

    # Arrêter si la page est vide ou si le total est atteint
    if not stations_page or len(toutes_stations) >= total_annonce:
        break

    # Passer à la page suivante
    offset += taille_page

print("Total annoncé :", total_annonce)
print("Total collecté :", len(toutes_stations))

Offset 0 : 100 stations reçues
Offset 100 : 100 stations reçues
Total annoncé : 200
Total collecté : 200


In [6]:
ids = [station["id"] for station in toutes_stations]

print("Total annoncé :", total_annonce)
print("Stations collectées :", len(toutes_stations))
print("Identifiants uniques :", len(set(ids)))
print("Doublons :", len(ids) - len(set(ids)))

assert len(toutes_stations) == total_annonce, (
    "Collecte incomplète : nombre de stations différent du total annoncé."
)

assert len(ids) == len(set(ids)), (
    "Des identifiants sont présents plusieurs fois."
)

Total annoncé : 200
Stations collectées : 200
Identifiants uniques : 200
Doublons : 0


In [7]:
print("Dossier de travail :", Path.cwd())

Dossier de travail : C:\Users\bowma\optifleet-logistics\notebooks


In [8]:
dossier_raw = Path("../data/raw")
dossier_raw.mkdir(parents=True, exist_ok=True)

print("Dossier de sauvegarde :", dossier_raw.resolve())

Dossier de sauvegarde : C:\Users\bowma\optifleet-logistics\data\raw


In [9]:
# Enregistrer l'heure actuelle en UTC
date_collecte = datetime.now(timezone.utc)

# Construire le dictionnaire à sauvegarder
collecte_brute = {
    "date_collecte": date_collecte.isoformat(),
    "source": URL,
    "filtre": params_page["where"],
    "total_count": total_annonce,
    "nombre_collecte": len(toutes_stations),
    "results": toutes_stations
}

In [10]:
# Créer un nom de fichier unique avec la date et l'heure UTC
nom_fichier = (
    f"carburants_69_{date_collecte.strftime('%Y-%m-%d_%H%M%S')}.json"
)

# Construire le chemin complet
chemin_fichier = dossier_raw / nom_fichier

# Écrire le dictionnaire dans un fichier JSON
with open(chemin_fichier, "w", encoding="utf-8") as fichier:
    json.dump(collecte_brute, fichier, ensure_ascii=False, indent=2)

print("Fichier enregistré :", chemin_fichier.resolve())

Fichier enregistré : C:\Users\bowma\optifleet-logistics\data\raw\carburants_69_2026-10-15_080144.json


In [11]:
with open(chemin_fichier, "r", encoding="utf-8") as fichier:
    verification = json.load(fichier)

print("Date de collecte :", verification["date_collecte"])
print("Stations enregistrées :", len(verification["results"]))
print("Fichier existe :", chemin_fichier.exists())
print("Taille du fichier :", chemin_fichier.stat().st_size, "octets")

Date de collecte : 2026-10-15T08:01:44.944796+00:00
Stations enregistrées : 200
Fichier existe : True
Taille du fichier : 682093 octets


In [12]:
# Récupérer les identifiants des stations sauvegardées
ids = [station["id"] for station in verification["results"]]

# Comparer les quantités
print("Total annoncé :", verification["total_count"])
print("Stations sauvegardées :", len(ids))
print("Stations uniques :", len(set(ids)))
print("Doublons :", len(ids) - len(set(ids)))

Total annoncé : 200
Stations sauvegardées : 200
Stations uniques : 200
Doublons : 0


## C3 - Construction du DataFrame Pandas

Objectif : transformer les données JSON brutes
en tableau Pandas pour faciliter leur analyse.

Source : fichier JSON sauvegardé dans data/raw/.

In [13]:
import pandas as pd

# Transformer la liste de dictionnaires en DataFrame
df_brut = pd.DataFrame(verification["results"])

# Afficher les dimensions du tableau
print("Dimensions :", df_brut.shape)

# Afficher les cinq premières lignes
display(df_brut.head())

Dimensions : (200, 47)


,id,latitude,longitude,cp,pop,adresse,ville,horaires,services,prix,...,carburants_indisponibles,carburants_rupture_temporaire,carburants_rupture_definitive,horaires_automate_24_24,services_service,departement,code_departement,region,code_region,horaires_jour
0,69002006,4573500,481800,69002,R,84 QUAI PERRACHE,Lyon,"{""@automate-24-24"": """", ""jour"": [{""@id"": ""1"", ...","{""service"": [""Boutique alimentaire"", ""Boutique...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, E85, GPLc]",None,None,Non,"[Boutique alimentaire, Boutique non alimentair...",Rhône,69,Auvergne-Rhône-Alpes,84,"Lundi00.00-23.59, Mardi00.00-23.59, Mercredi00..."
1,69003012,4574700,486700,69003,R,65 COURS ALBERT THOMAS,Lyon,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Boutique alimentaire"", ""Boutique...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[E85, GPLc, SP98]",None,E85;GPLc;SP98,Oui,"[Boutique alimentaire, Boutique non alimentair...",Rhône,69,Auvergne-Rhône-Alpes,84,"Automate-24-24, Lundi06.00-21.00, Mardi06.00-2..."
2,69003013,4575446.02,485283.65475131,69003,R,258 rue Garibaldi,Lyon,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Toilettes publiques"", ""Boutique ...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[E85, SP98]",None,E85;SP98,Oui,"[Toilettes publiques, Boutique alimentaire, Bo...",Rhône,69,Auvergne-Rhône-Alpes,84,"Automate-24-24, Lundi06.00-22.00, Mardi06.00-2..."
3,69004002,4577600,482100,69004,R,22 Rue Philippe de Lassalle,Lyon,"{""@automate-24-24"": """", ""jour"": [{""@id"": ""1"", ...",None,"[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, GPLc]",None,None,Non,None,Rhône,69,Auvergne-Rhône-Alpes,84,None
4,69004016,4577900,482800,69004,R,97 RUE DENFERT ROCHEREAU,Lyon,"{""@automate-24-24"": ""1"", ""jour"": [{""@id"": ""1"",...","{""service"": [""Toilettes publiques"", ""Boutique ...","[{""@nom"": ""Gazole"", ""@id"": ""1"", ""@maj"": ""2026-...",...,"[SP95, E85, GPLc]",SP95;E85;GPLc,None,Oui,"[Toilettes publiques, Boutique alimentaire, St...",Rhône,69,Auvergne-Rhône-Alpes,84,Automate-24-24


In [14]:
# Compter toutes les stations
nb_stations = len(df_brut)

# Compter les stations sans prix Gazole
nb_prix_manquants = df_brut["gazole_prix"].isna().sum()

# Calculer le pourcentage de prix manquants
pourcentage_manquants = (
    nb_prix_manquants / nb_stations
) * 100

print("Stations collectées :", nb_stations)
print("Prix Gazole manquants :", nb_prix_manquants)
print(f"Pourcentage manquant : {pourcentage_manquants:.2f} %")

Stations collectées : 200
Prix Gazole manquants : 8
Pourcentage manquant : 4.00 %


### C3.2 - Sélection des colonnes utiles

Objectif : créer un DataFrame de travail contenant
uniquement les colonnes nécessaires à l'analyse
des prix du Gazole pour OptiFleet Logistics.

Le DataFrame brut sera conservé sans modification.

In [15]:
# Définir les colonnes nécessaires à l'analyse
colonnes_utiles = [
    "id",
    "adresse",
    "ville",
    "cp",
    "geom",
    "gazole_prix",
    "gazole_maj",
    "code_departement"
]

# Créer un nouveau DataFrame à partir des colonnes sélectionnées
df = df_brut[colonnes_utiles].copy()

# Vérifier les dimensions
print("Dimensions du DataFrame brut :", df_brut.shape)
print("Dimensions du DataFrame de travail :", df.shape)

# Afficher les cinq premières stations
display(df.head())

Dimensions du DataFrame brut : (200, 47)
Dimensions du DataFrame de travail : (200, 8)


,id,adresse,ville,cp,geom,gazole_prix,gazole_maj,code_departement
0,69002006,84 QUAI PERRACHE,Lyon,69002,"{'lon': 4.818, 'lat': 45.735}",2.489,2026-10-14T15:21:57+00:00,69
1,69003012,65 COURS ALBERT THOMAS,Lyon,69003,"{'lon': 4.867, 'lat': 45.747}",2.429,2026-10-14T20:50:00+00:00,69
2,69003013,258 rue Garibaldi,Lyon,69003,"{'lon': 4.8528365475131, 'lat': 45.7544602}",2.449,2026-10-05T21:45:00+00:00,69
3,69004002,22 Rue Philippe de Lassalle,Lyon,69004,"{'lon': 4.821, 'lat': 45.776}",2.343,2026-10-15T07:32:00+00:00,69
4,69004016,97 RUE DENFERT ROCHEREAU,Lyon,69004,"{'lon': 4.828, 'lat': 45.779}",2.469,2026-10-14T21:00:00+00:00,69


In [16]:
# Vérifier que les prix manquants ont été conservés
print("Stations dans df :", len(df))
print("Prix Gazole manquants :", df["gazole_prix"].isna().sum())

# Afficher les stations dont le prix est absent
display(
    df.loc[
        df["gazole_prix"].isna(),
        ["id", "ville", "adresse", "gazole_prix"]
    ]
)

Stations dans df : 200
Prix Gazole manquants : 8


,id,ville,adresse,gazole_prix
18,69100022,Bron,283 RTE DE GENAS,NaN
53,69200009,Vénissieux,RTE DE CORBAS - RD 57,NaN
72,69250009,Neuville-sur-Saône,28 AVENUE BURDEAU,NaN
109,69380014,Lozanne,370 ROUTE DE LA TOUR DE SALVAGNY,NaN
121,69400016,Arnas,2315 ROUTE DE LONGSARD,NaN
173,69700002,Loire-sur-Rhône,1293 Route de Beaucaire (Ex.45 R.N. 86),NaN
194,69870001,Lamure-sur-Azergues,890 rue centrale,NaN
197,69930001,Saint-Laurent-de-Chamousset,La Croix Blanche,NaN


### C3.3 - Gestion des dates et fraîcheur des prix

Objectif : calculer l'ancienneté de chaque prix du Gazole
par rapport à la date de collecte du fichier JSON.

Les calculs sont effectués en UTC pour éviter
les erreurs liées aux fuseaux horaires.

In [17]:
# 1. Lire l'horodatage de collecte depuis le JSON
horodatage_collecte = pd.to_datetime(
    verification["date_collecte"],
    utc=True
)

# 2. Ajouter l'horodatage UTC à toutes les stations
df["horodatage_collecte"] = horodatage_collecte

# 3. Convertir en heure de Paris pour obtenir la date métier
df["date_collecte"] = (
    df["horodatage_collecte"]
    .dt.tz_convert("Europe/Paris")
    .dt.date
)

# 4. Vérifier les résultats
display(
    df[
        ["id", "ville", "horodatage_collecte", "date_collecte"]
    ].head())

,id,ville,horodatage_collecte,date_collecte
0,69002006,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
1,69003012,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
2,69003013,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
3,69004002,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15
4,69004016,Lyon,2026-10-15 08:01:44.944796+00:00,2026-10-15


In [18]:
# 1. Convertir gazole_maj en datetime avec fuseau UTC
df["gazole_maj"] = pd.to_datetime(
    df["gazole_maj"],
    errors="coerce",
    utc=True
)

# 2. Calculer la différence entre la collecte et la mise à jour
difference = (
    df["horodatage_collecte"] - df["gazole_maj"]
)

# 3. Convertir cette différence en heures
df["age_prix_heures"] = (
    difference.dt.total_seconds() / 3600
)

# 4. Vérifier les types
print("Type gazole_maj :", df["gazole_maj"].dtype)
print("Type horodatage_collecte :", df["horodatage_collecte"].dtype)

# 5. Afficher les résultats
display(
    df[
        [
            "id",
            "ville",
            "gazole_prix",
            "gazole_maj",
            "horodatage_collecte",
            "age_prix_heures"
        ]
    ].head()
)

Type gazole_maj : datetime64[ns, UTC]
Type horodatage_collecte : datetime64[ns, UTC]


,id,ville,gazole_prix,gazole_maj,horodatage_collecte,age_prix_heures
0,69002006,Lyon,2.489,2026-10-14 15:21:57+00:00,2026-10-15 08:01:44.944796+00:00,16.663318
1,69003012,Lyon,2.429,2026-10-14 20:50:00+00:00,2026-10-15 08:01:44.944796+00:00,11.195818
2,69003013,Lyon,2.449,2026-10-05 21:45:00+00:00,2026-10-15 08:01:44.944796+00:00,226.279151
3,69004002,Lyon,2.343,2026-10-15 07:32:00+00:00,2026-10-15 08:01:44.944796+00:00,0.495818
4,69004016,Lyon,2.469,2026-10-14 21:00:00+00:00,2026-10-15 08:01:44.944796+00:00,11.029151


In [19]:
# Identifier les dates de mise à jour postérieures à la collecte
prix_futurs = df[df["age_prix_heures"] < 0]

print("Nombre de prix datés après la collecte :", len(prix_futurs))

display(
    prix_futurs[
        ["id", "ville", "gazole_maj",
         "date_collecte", "age_prix_heures"]
    ]
)

Nombre de prix datés après la collecte : 10


,id,ville,gazole_maj,date_collecte,age_prix_heures
33,69130005,Écully,2026-10-15 08:14:35+00:00,2026-10-15,-0.213904
34,69132001,Écully,2026-10-15 08:26:00+00:00,2026-10-15,-0.404182
117,69400008,Gleizé,2026-10-15 09:27:00+00:00,2026-10-15,-1.420849
120,69400015,Gleizé,2026-10-15 08:53:27+00:00,2026-10-15,-0.861682
152,69550002,Amplepuis,2026-10-15 08:48:21+00:00,2026-10-15,-0.776682
168,69658001,Villefranche-sur-Saône,2026-10-15 08:17:24+00:00,2026-10-15,-0.260849
182,69800010,Saint-Priest,2026-10-15 09:07:24+00:00,2026-10-15,-1.094182
185,69800014,Saint-Priest,2026-10-15 09:00:00+00:00,2026-10-15,-0.970849
192,69830003,Saint-Georges-de-Reneins,2026-10-15 08:37:26+00:00,2026-10-15,-0.594738
199,69970001,Chaponnay,2026-10-15 09:23:57+00:00,2026-10-15,-1.370015


In [20]:
# Vérifier l'ampleur des écarts temporels négatifs

print(
    "Âge minimum observé (h) :",
    df["age_prix_heures"].min()
)

print(
    "Dates futures de plus de 2 heures :",
    (df["age_prix_heures"] < -2).sum()
)

Âge minimum observé (h) : -1.4208486677777779
Dates futures de plus de 2 heures : 0


In [21]:
# Choisir une station présentant une date future apparente
id_station = 69400008

# Retrouver sa ligne dans les données brutes
station_brute = df_brut.loc[
    df_brut["id"] == id_station
]

# Afficher les informations d'origine
print("Station :", id_station)
print("Date gazole_maj :", station_brute["gazole_maj"].iloc[0])
print("Champ prix brut :")
print(station_brute["prix"].iloc[0])

Station : 69400008
Date gazole_maj : 2026-10-15T09:27:00+00:00
Champ prix brut :
[{"@nom": "Gazole", "@id": "1", "@maj": "2026-10-15 09:27:00", "@valeur": "2.399"}, {"@nom": "E85", "@id": "3", "@maj": "2026-10-15 09:27:00", "@valeur": "0.869"}, {"@nom": "E10", "@id": "5", "@maj": "2026-10-15 09:27:00", "@valeur": "2.209"}, {"@nom": "SP98", "@id": "6", "@maj": "2026-10-15 09:27:00", "@valeur": "2.300"}]


### Conclusion provisoire - Anomalie temporelle

Lors de la collecte du 15 octobre 2026, 10 stations sur 200
présentent une date de mise à jour apparemment postérieure
à la collecte.

Pour la station 69400008, le champ brut `prix` indique
`2026-10-15 09:27:00` sans fuseau horaire, tandis que
`gazole_maj` indique `2026-10-15T09:27:00+00:00`.

Cette observation est compatible avec l'hypothèse d'une
heure locale interprétée comme UTC, mais ne suffit pas
à confirmer une erreur de fuseau horaire.

**Décision :** conserver les horodatages d'origine,
signaler les dates futures apparentes et ne pas appliquer
de correction automatique sans validation supplémentaire.

### C3.4 - Classification de la fraîcheur des prix

Objectif : identifier les prix suffisamment récents
pour être utilisés dans les recommandations OptiFleet.

Règle métier :
- Âge compris entre 0 et 48 heures : récent.
- Âge supérieur à 48 heures : ancien.
- Âge négatif : anomalie temporelle.
- Date manquante : fraîcheur inconnue.

In [22]:
# 1. Un prix est récent si son âge est compris entre 0 et 48 h
condition_recent = df["age_prix_heures"].between(0, 48)

# 2. Le prix doit également être renseigné
condition_prix_present = df["gazole_prix"].notna()

# 3. Créer l'indicateur final
df["prix_recent_48h"] = (
    condition_recent & condition_prix_present
)

# 4. Compter les stations éligibles
print(df["prix_recent_48h"].value_counts(dropna=False))

prix_recent_48h
True     121
False     79
Name: count, dtype: int64


In [24]:
# 1. Compter les prix récents et renseignés
nb_recents = df["prix_recent_48h"].sum()

# 2. Compter les prix anciens et renseignés
nb_anciens = (
    (df["age_prix_heures"] > 48)
    & df["gazole_prix"].notna()
).sum()

# 3. Compter les dates futures apparentes
nb_futurs = (
    (df["age_prix_heures"] < 0)
    & df["gazole_prix"].notna()
).sum()

# 4. Compter les prix Gazole manquants
nb_prix_manquants = df["gazole_prix"].isna().sum()

# 5. Compter les dates de mise à jour manquantes
# uniquement pour les stations avec un prix
nb_dates_manquantes = (
    df["age_prix_heures"].isna()
    & df["gazole_prix"].notna()
).sum()

# 6. Afficher les résultats
print("Prix récents (0 - 48 h) :", nb_recents)
print("Prix anciens (> 48 h) :", nb_anciens)
print("Dates futures apparentes :", nb_futurs)
print("Prix Gazole manquants :", nb_prix_manquants)
print("Dates manquantes avec prix :", nb_dates_manquantes)

# 7. Vérifier que chaque station appartient
# à une seule catégorie
total_controle = (
    nb_recents
    + nb_anciens
    + nb_futurs
    + nb_prix_manquants
    + nb_dates_manquantes
)

print("-----------------------------")
print("Total contrôlé :", total_controle)
print("Total stations :", len(df))
print("Contrôle réussi :", total_controle == len(df))

Prix récents (0 - 48 h) : 121
Prix anciens (> 48 h) : 61
Dates futures apparentes : 10
Prix Gazole manquants : 8
Dates manquantes avec prix : 0
-----------------------------
Total contrôlé : 200
Total stations : 200
Contrôle réussi : True


## C4 - Contrôles qualité et préparation des données

### C4.1 - Contrôle des prix du Gazole

Objectif : vérifier que les prix sont renseignés,
numériques et situés dans une plage plausible.

Règle provisoire : entre 0,50 et 4,00 €/L.

Aucune ligne ne sera supprimée avant
l'analyse des anomalies.

In [25]:
# Vérifier le type de données de la colonne
print("Type initial :", df["gazole_prix"].dtype)

# Compter les valeurs manquantes
print("Prix manquants :", df["gazole_prix"].isna().sum())

# Afficher les statistiques descriptives
display(df["gazole_prix"].describe())

Type initial : float64
Prix manquants : 8


count    192.000000
mean       2.363677
std        0.088978
min        2.240000
25%        2.261250
50%        2.359000
75%        2.426750
max        2.649000
Name: gazole_prix, dtype: float64

In [26]:
df["gazole_prix"].between(0.50, 4.00)

0       True
1       True
2       True
3       True
4       True
       ...  
195     True
196     True
197    False
198     True
199     True
Name: gazole_prix, Length: 200, dtype: bool

### C4.2 - Détection des prix atypiques avec l'IQR

Objectif : identifier les prix potentiellement atypiques
par rapport à la distribution observée.

Méthode : quartiles Q1 et Q3, écart interquartile
et règle des 1,5 × IQR.

Les prix signalés seront examinés avant
toute décision de correction ou d'exclusion.

In [27]:
# Calculer les quartiles
Q1 = df["gazole_prix"].quantile(0.25)
Q3 = df["gazole_prix"].quantile(0.75)

# Calculer l'écart interquartile
IQR = Q3 - Q1

# Définir les bornes de détection
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

# Afficher les résultats
print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR :", IQR)
print("Borne inférieure :", borne_inf)
print("Borne supérieure :", borne_sup)

Q1 : 2.26125
Q3 : 2.42675
IQR : 0.1655000000000002
Borne inférieure : 2.013
Borne supérieure : 2.6750000000000007


In [28]:
# Créer une condition de détection des prix atypiques
condition_atypique = (
    (df["gazole_prix"] < borne_inf) |
    (df["gazole_prix"] > borne_sup)
)

# Ajouter un indicateur booléen
df["prix_atypique_iqr"] = condition_atypique

# Compter les prix atypiques
print("Prix atypiques :", df["prix_atypique_iqr"].sum())

# Afficher les stations concernées
display(
    df.loc[
        df["prix_atypique_iqr"],
        ["id", "ville", "gazole_prix", "gazole_maj"]
    ]
)

Prix atypiques : 0


,id,ville,gazole_prix,gazole_maj


### C4.3 - Synthèse des contrôles qualité

Objectif : regrouper les résultats des contrôles
dans un tableau de synthèse reproductible.

Contrôles réalisés :
- Unicité des identifiants.
- Présence et plausibilité des prix.
- Détection des valeurs atypiques (IQR).
- Fraîcheur et cohérence des dates.

In [29]:
# Calculer les indicateurs de qualité
controles = {
    "Stations collectées": len(df),
    "Identifiants uniques": df["id"].nunique(),
    "Doublons d'identifiants": df["id"].duplicated().sum(),

    "Prix Gazole manquants": nb_prix_manquants,

    "Prix hors bornes": (
        df["gazole_prix"].notna()
        & ~df["gazole_prix"].between(0.50, 4.00)
    ).sum(),

    "Prix atypiques IQR": df["prix_atypique_iqr"].sum(),

    "Prix récents (0-48 h)": nb_recents,
    "Prix anciens (>48 h)": nb_anciens,
    "Dates futures apparentes": nb_futurs,
    "Dates manquantes avec prix": nb_dates_manquantes
}

# Présenter les contrôles sous forme de tableau
df_controles = pd.DataFrame(
    list(controles.items()),
    columns=["Contrôle", "Nombre"]
)

display(df_controles)

,Contrôle,Nombre
0,Stations collectées,200
1,Identifiants uniques,200
2,Doublons d'identifiants,0
3,Prix Gazole manquants,8
4,Prix hors bornes,0
5,Prix atypiques IQR,0
6,Prix récents (0-48 h),121
7,Prix anciens (>48 h),61
8,Dates futures apparentes,10
9,Dates manquantes avec prix,0


### C4.4 - Préparation des données pour l'export CSV

Objectif : préparer un fichier tabulaire facilement
importable dans BigQuery.

Transformations :
- Extraction de la latitude et de la longitude depuis geom.
- Conservation des indicateurs de qualité.
- Conservation des dates en UTC.
- Export des données au format CSV.

In [30]:
# Créer une copie pour préparer l'export
df_export = df.copy()

# Extraire la latitude et la longitude
df_export["latitude"] = df_export["geom"].apply(
    lambda position: position.get("lat")
    if isinstance(position, dict) else None
)

df_export["longitude"] = df_export["geom"].apply(
    lambda position: position.get("lon")
    if isinstance(position, dict) else None
)

# Vérifier les coordonnées
display(
    df_export[
        ["id", "ville", "latitude", "longitude"]
    ].head()
)

,id,ville,latitude,longitude
0,69002006,Lyon,45.73500,4.818000
1,69003012,Lyon,45.74700,4.867000
2,69003013,Lyon,45.75446,4.852837
3,69004002,Lyon,45.77600,4.821000
4,69004016,Lyon,45.77900,4.828000


In [31]:
print("Latitudes manquantes :", df_export["latitude"].isna().sum())
print("Longitudes manquantes :", df_export["longitude"].isna().sum())

print("Type latitude :", df_export["latitude"].dtype)
print("Type longitude :", df_export["longitude"].dtype)

Latitudes manquantes : 0
Longitudes manquantes : 0
Type latitude : float64
Type longitude : float64


### C4.5  Contrôle de validité des coordonnées

Objectif : vérifier que les coordonnées géographiques
respectent les bornes mondiales de latitude et longitude.

- Latitude : entre -90 et 90.
- Longitude : entre -180 et 180.

Ce contrôle ne garantit pas encore que les stations
se trouvent dans la Métropole de Lyon.

In [32]:
# Vérifier que les coordonnées respectent les bornes géographiques
coordonnees_valides = (
    df_export["latitude"].between(-90, 90)
    & df_export["longitude"].between(-180, 180)
)

# Compter les stations valides et invalides
print("Coordonnées valides :", coordonnees_valides.sum())
print("Coordonnées invalides :", (~coordonnees_valides).sum())

# Afficher les éventuelles stations problématiques
display(
    df_export.loc[
        ~coordonnees_valides,
        ["id", "ville", "latitude", "longitude"]
    ]
)

Coordonnées valides : 200
Coordonnées invalides : 0


,id,ville,latitude,longitude


In [33]:
# Retirer la colonne geom du DataFrame d'export
df_export = df_export.drop(columns=["geom"])

# Vérifier les dimensions et les colonnes
print("Dimensions :", df_export.shape)
print("Colonnes :", df_export.columns.tolist())

display(df_export.head())

Dimensions : (200, 14)
Colonnes : ['id', 'adresse', 'ville', 'cp', 'gazole_prix', 'gazole_maj', 'code_departement', 'horodatage_collecte', 'date_collecte', 'age_prix_heures', 'prix_recent_48h', 'prix_atypique_iqr', 'latitude', 'longitude']


,id,adresse,ville,cp,gazole_prix,gazole_maj,code_departement,horodatage_collecte,date_collecte,age_prix_heures,prix_recent_48h,prix_atypique_iqr,latitude,longitude
0,69002006,84 QUAI PERRACHE,Lyon,69002,2.489,2026-10-14 15:21:57+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,16.663318,True,False,45.73500,4.818000
1,69003012,65 COURS ALBERT THOMAS,Lyon,69003,2.429,2026-10-14 20:50:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.195818,True,False,45.74700,4.867000
2,69003013,258 rue Garibaldi,Lyon,69003,2.449,2026-10-05 21:45:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,226.279151,False,False,45.75446,4.852837
3,69004002,22 Rue Philippe de Lassalle,Lyon,69004,2.343,2026-10-15 07:32:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,0.495818,True,False,45.77600,4.821000
4,69004016,97 RUE DENFERT ROCHEREAU,Lyon,69004,2.469,2026-10-14 21:00:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.029151,True,False,45.77900,4.828000


### C4.7 - Export des données préparées en CSV

Objectif : sauvegarder le DataFrame contrôlé
dans le dossier data/processed/.

Le fichier CSV servira à l'importation dans BigQuery.

Les données brutes JSON sont conservées séparément
dans data/raw/.

In [34]:
# Définir le dossier de destination
dossier_processed = Path("../data/processed")

# Créer le dossier s'il n'existe pas
dossier_processed.mkdir(parents=True, exist_ok=True)

# Construire le nom du CSV à partir de celui du JSON
nom_csv = chemin_fichier.stem + ".csv"

# Construire le chemin complet
chemin_csv = dossier_processed / nom_csv

# Exporter le DataFrame
df_export.to_csv(
    chemin_csv,
    index=False,
    encoding="utf-8"
)

print("CSV enregistré :", chemin_csv.resolve())

CSV enregistré : C:\Users\bowma\optifleet-logistics\data\processed\carburants_69_2026-10-15_080144.csv


In [35]:
# Relire le fichier CSV
df_verification_csv = pd.read_csv(
    chemin_csv,
    dtype={
        "cp": "string",
        "code_departement": "string"
    }
)

# Comparer les dimensions
print("Dimensions avant export :", df_export.shape)
print("Dimensions après relecture :", df_verification_csv.shape)

# Vérifier l'existence du fichier
print("Fichier existe :", chemin_csv.exists())

# Vérifier les identifiants uniques
print("Stations uniques :", df_verification_csv["id"].nunique())

# Afficher les premières lignes
display(df_verification_csv.head())

Dimensions avant export : (200, 14)
Dimensions après relecture : (200, 14)
Fichier existe : True
Stations uniques : 200


,id,adresse,ville,cp,gazole_prix,gazole_maj,code_departement,horodatage_collecte,date_collecte,age_prix_heures,prix_recent_48h,prix_atypique_iqr,latitude,longitude
0,69002006,84 QUAI PERRACHE,Lyon,69002,2.489,2026-10-14 15:21:57+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,16.663318,True,False,45.73500,4.818000
1,69003012,65 COURS ALBERT THOMAS,Lyon,69003,2.429,2026-10-14 20:50:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.195818,True,False,45.74700,4.867000
2,69003013,258 rue Garibaldi,Lyon,69003,2.449,2026-10-05 21:45:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,226.279151,False,False,45.75446,4.852837
3,69004002,22 Rue Philippe de Lassalle,Lyon,69004,2.343,2026-10-15 07:32:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,0.495818,True,False,45.77600,4.821000
4,69004016,97 RUE DENFERT ROCHEREAU,Lyon,69004,2.469,2026-10-14 21:00:00+00:00,69,2026-10-15 08:01:44.944796+00:00,2026-10-15,11.029151,True,False,45.77900,4.828000


### C4.8 - Premier KPI métier : écart médiane − P10

**Objectif :** calculer l'écart entre le prix médian du Gazole
et le 10e percentile (P10).

Cet écart sert à estimer les économies potentielles
pour OptiFleet Logistics.

**Hypothèse :** consommation mensuelle de 10 400 litres.

**Limite :** le calcul porte encore sur le département 69.
Il sera recalculé sur la Métropole de Lyon après C5.

In [37]:
# C4.8 — Calculer le premier KPI métier

# Conserver uniquement les prix renseignés pour le calcul
prix_valides = df["gazole_prix"].dropna()

# Calculer la médiane et le P10
mediane = prix_valides.median()
p10 = prix_valides.quantile(0.10)

# Calculer l'écart
ecart_prix = mediane - p10

# Estimer les économies mensuelles
volume_mensuel = 10400
economie_mensuelle = ecart_prix * volume_mensuel

# Afficher les résultats
print(f"Médiane : {mediane:.3f} €/L")
print(f"P10 : {p10:.3f} €/L")
print(f"Écart médiane - P10 : {ecart_prix:.3f} €/L")
print(f"Économie mensuelle théorique : {economie_mensuelle:.0f} €")

Médiane : 2.359 €/L
P10 : 2.250 €/L
Écart médiane - P10 : 0.109 €/L
Économie mensuelle théorique : 1134 €


### C4.9 - Test de sensibilité du seuil de fraîcheur

**Objectif :** comparer la couverture des stations
selon trois seuils de fraîcheur :

- 48 heures (2 jours)
- 72 heures (3 jours)
- 168 heures (7 jours)

Une station est éligible si son prix Gazole est renseigné
et si son âge est compris entre 0 et le seuil étudié.

Les dates futures apparentes restent exclues provisoirement.

In [38]:
# C4.9 — Tester différents seuils de fraîcheur

# Compter les stations dont le prix Gazole est renseigné
prix_renseignes = df["gazole_prix"].notna()
total_prix_renseignes = prix_renseignes.sum()

# Tester trois seuils
for seuil in [48, 72, 168]:

    # Sélectionner les prix renseignés et suffisamment récents
    eligibles = (
        prix_renseignes
        & df["age_prix_heures"].between(0, seuil)
    )

    # Compter les stations éligibles
    nombre = eligibles.sum()

    # Calculer leur proportion
    pourcentage = nombre / total_prix_renseignes * 100

    print(
        f"Seuil {seuil} h : "
        f"{nombre} stations ({pourcentage:.1f} %)"
    )

Seuil 48 h : 121 stations (63.0 %)
Seuil 72 h : 133 stations (69.3 %)
Seuil 168 h : 153 stations (79.7 %)


### C4 - Conclusion des analyses métier

**Premier KPI :**

- Prix médian du Gazole : 2,359 €/L.
- P10 : 2,250 €/L.
- Écart médiane − P10 : 0,109 €/L.
- Économie mensuelle théorique : environ 1 134 €,
  sur la base de 10 400 litres par mois.

**Test de sensibilité :**

- 48 h : 121 stations éligibles (63,0 %).
- 72 h : 133 stations éligibles (69,3 %).
- 168 h : 153 stations éligibles (79,7 %).

**Choix provisoire :** retenir 48 h pour privilégier
la fraîcheur des prix et conserver 72 h comme scénario
alternatif.

**Limites :**

- Analyses réalisées sur le département 69.
- Le périmètre Métropole de Lyon reste à appliquer.
- Les anomalies temporelles apparentes restent à examiner.
- Les économies calculées sont théoriques.

**Prochaine étape :** C5 — Filtrage géographique
sur les communes de la Métropole de Lyon.

# C5 - Délimitation géographique : Métropole de Lyon

## Objectif

Identifier les stations-service situées dans la Métropole de Lyon
parmi les stations collectées dans le département 69.

## Méthodologie

1. Obtenir une liste officielle des communes de la Métropole.
2. Comparer les communes des stations avec cette liste.
3. Identifier les stations appartenant au périmètre.
4. Contrôler les communes non reconnues.
5. Créer un DataFrame dédié à l'analyse métier.

**Important :** conserver toutes les stations dans les données brutes.
Le filtrage géographique sera appliqué uniquement aux données d'analyse.

In [36]:
# C5.1 - Explorer les communes présentes dans nos données

# Compter les noms de communes différents
nb_communes = df["ville"].nunique()

print("Nombre de noms de communes distincts :", nb_communes)

# Compter le nombre de stations par commune
stations_par_commune = (
    df["ville"]
    .value_counts()
    .rename_axis("commune")
    .reset_index(name="nombre_stations")
)

# Afficher le tableau complet
display(stations_par_commune)

Nombre de noms de communes distincts : 99


,commune,nombre_stations
0,Lyon,19
1,Saint-Priest,10
2,Villefranche-sur-Saône,8
3,Bron,6
4,Villeurbanne,6
...,...,...
94,Beaujeu,1
95,Lantignié,1
96,Quincié-en-Beaujolais,1
97,Chénelette,1


## C5.2 - Récupération du référentiel officiel

Objectif : récupérer les communes de la Métropole de Lyon
à partir de l'API Géo.

Nous utiliserons les codes INSEE pour fiabiliser
le périmètre géographique.

Les noms de communes seront conservés pour
l'affichage et les contrôles.

In [39]:
# C5.2 — Récupérer les communes de la Métropole de Lyon

# 1. Identifier la Métropole de Lyon
code_metropole = "200046977"

# 2. Construire l'URL de l'API Géo
url_communes = (
    f"https://geo.api.gouv.fr/epcis/{code_metropole}/communes"
)

# 3. Demander les champs nécessaires
parametres = {
    "fields": "nom,code,codesPostaux",
    "format": "json"
}

# 4. Envoyer la requête
reponse = requests.get(
    url_communes,
    params=parametres,
    timeout=30
)

# 5. Vérifier le statut HTTP
print("Statut HTTP :", reponse.status_code)
reponse.raise_for_status()

# 6. Convertir le JSON en DataFrame
communes_json = reponse.json()
df_communes = pd.DataFrame(communes_json)

# 7. Examiner le résultat
print("Nombre de communes :", len(df_communes))
print("Colonnes disponibles :", df_communes.columns.tolist())

display(df_communes.head())

Statut HTTP : 200
Nombre de communes : 58
Colonnes disponibles : ['nom', 'code', 'codesPostaux']


,nom,code,codesPostaux
0,Albigny-sur-Saône,69003,[69250]
1,Bron,69029,[69500]
2,Cailloux-sur-Fontaines,69033,[69270]
3,Caluire-et-Cuire,69034,[69300]
4,Champagne-au-Mont-d'Or,69040,[69410]


In [40]:
# C5.2 — Vérifier la qualité du référentiel

# Vérifier le nombre de communes
assert len(df_communes) == 58, "Le nombre de communes est inattendu"

# Vérifier les codes INSEE uniques
print("Communes :", len(df_communes))
print("Codes INSEE uniques :", df_communes["code"].nunique())

# Vérifier les valeurs manquantes
print("\nValeurs manquantes :")
print(df_communes.isna().sum())

# Vérifier les doublons de codes INSEE
print(
    "\nDoublons de codes INSEE :",
    df_communes["code"].duplicated().sum()
)

Communes : 58
Codes INSEE uniques : 58

Valeurs manquantes :
nom             0
code            0
codesPostaux    0
dtype: int64

Doublons de codes INSEE : 0
